# ベースライン学習（検証: 時系列CV）

- **使う特徴量**: 34（3C3）＋ テキストメタ 4 列 ＝ **38 特徴**（movie_info_and_title_meta。train_text_experiments で CV 最良だった設定）。
- **検証**: 時系列CV（2013〜2016 を val）。提出は全 train で 1 本学習して test を予測。

In [6]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from preprocess import load_train_test
from feature_engineering import create_features

print("Setup complete.")

Setup complete.


In [7]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [8]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [9]:
train = create_features(train)
test = create_features(test)

# モデルにそのまま使う列のうち、object は category に（欠損は "missing"）
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
# 数値で欠損がある列は train の中央値で埋める
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成完了.")

特徴量作成完了.


In [10]:
# 3C3 パターン: 時系列TE（critic_name, production_company）＋ TE 3ビン ＋ runtime_bin, movie_age_bin, release_decade
def _ts_te_col(df_tr, df_te, col, target_name="target", m=10):
    """時系列TE: tr は「その行より前」のみで平均、te は tr のカテゴリ別スムージング平均でマッピング。"""
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, te_arr

for col in ["critic_name", "production_company"]:
    if col not in train.columns:
        continue
    st, ta = _ts_te_col(train, test, col, "target", m=10)
    train[f"{col}_te_ts"] = st.reindex(train.index).fillna(train["target"].mean())
    test[f"{col}_te_ts"] = ta

for c in ["critic_name_te_ts", "production_company_te_ts"]:
    if c not in train.columns:
        continue
    train[c + "_bin"] = pd.cut(train[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
    test[c + "_bin"] = pd.cut(test[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)

train["runtime_bin"] = pd.cut(train["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
test["runtime_bin"] = pd.cut(test["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
def _movie_age_bin(x):
    if pd.isna(x) or x < 0: return 0
    if x < 365: return 1
    if x < 365 * 5: return 2
    if x < 365 * 20: return 3
    return 4
train["movie_age_bin"] = train["movie_age_days"].apply(_movie_age_bin)
test["movie_age_bin"] = test["movie_age_days"].apply(_movie_age_bin)
train["release_decade"] = (train["release_year"] // 10 * 10).fillna(1990)
test["release_decade"] = (test["release_year"] // 10 * 10).fillna(1990)
print("3C3 特徴量追加完了.")

3C3 特徴量追加完了.


In [ ]:
# テキストメタ追加（movie_info_and_title_meta: train_text_experiments で最良だった設定）
from experiment_encodings import movie_info_meta
movie_info_meta(train, pd.DataFrame(), test)
train["movie_title_len"] = train["movie_title"].astype(str).str.len()
test["movie_title_len"] = test["movie_title"].astype(str).str.len()
train["movie_title_word_count"] = train["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
test["movie_title_word_count"] = test["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
print("テキストメタ（movie_info_and_title_meta）追加完了.")

In [11]:
# 使う特徴量はノート内で列挙。3C3 ＋ テキストメタ（movie_info_and_title_meta）で 38 特徴。
FEATURES = [
    "rotten_tomatoes_link",
    "critic_name",
    "top_critic",
    "publisher_name",
    #映画情報
    "movie_title",
    "movie_info",
    "content_rating",
    "directors",
    "authors",
    "actors",
    "runtime",
    "production_company",
    #レビュー系
    "review_year",
    "review_month",
    "review_dayofweek",
    #リリース系
    "release_year",
    "release_month",
    "release_dayofweek",
    "movie_age_days",
    #カテゴリ特徴量
    "genre_Drama",
    "genre_Comedy",
    "genre_Action",
    "genre_Mystery",
    "genre_Fantasy",
    "genre_Romance",
    "genre_Horror",
    "genre_Documentary",
    # 3C3 追加
    "critic_name_te_ts",
    "production_company_te_ts",
    "critic_name_te_ts_bin",
    "production_company_te_ts_bin",
    "runtime_bin",
    "movie_age_bin",
    "release_decade",
    # テキストメタ（movie_info_and_title_meta）
    "movie_info_len",
    "movie_info_word_count",
    "movie_title_len",
    "movie_title_word_count",
]
print(f"使用特徴量: {len(FEATURES)}個")

使用特徴量: 34個


In [12]:
train

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,genre_Romance,genre_Horror,genre_Documentary,critic_name_te_ts,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.803451,0.489462,2.0,1.0,1.0,0,2010.0
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.820255,0.489491,2.0,1.0,1.0,0,2010.0
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.413455,0.489582,1.0,1.0,1.0,0,2010.0
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.360928,0.489637,1.0,1.0,1.0,0,2010.0
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.682583,0.489549,2.0,1.0,1.0,0,2010.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653502,653502,m/zulu_dawn,Brandon Judell,False,PopcornQ,2005-08-14,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.702341,0.562785,2.0,1.0,2.0,4,1970.0
653503,653503,m/zulu_dawn,Cole Smithey,False,ColeSmithey.com,2005-11-01,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.820266,0.599219,2.0,1.0,2.0,4,1970.0
653504,653504,m/zulu_dawn,Ken Hanke,False,"Mountain Xpress (Asheville, NC)",2007-03-07,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.711286,0.630048,2.0,1.0,2.0,4,1970.0
653505,653505,m/zulu_dawn,Dennis Schwartz,False,Dennis Schwartz Movie Reviews,2010-09-16,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.602209,0.656474,1.0,1.0,2.0,4,1970.0


In [13]:
# 時系列CVの分割は次のセルで実行

In [14]:
# 時系列CV: 検証年ごとに「その年より前で学習 → その年で検証」
# train の review_year 範囲に合わせて検証年を設定（test が 2017〜 なので 2013〜2016 などで検証）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [15]:
X = train[FEATURES]
y = train["target"].values
X_test = test[FEATURES]

lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

oof = np.zeros(len(train))
test_preds = []
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    test_preds.append(model.predict_proba(X_test)[:, 1])
    print(f"Fold{fold+1}: val_year={VAL_YEARS[fold]}, AUC={fold_auc:.4f}")

# 時系列CVでは oof が全行を埋めないため、CV AUC は fold 平均で表示
print(f"\nCV AUC (fold mean): {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

Fold1: val_year=2013, AUC=0.7600
Fold2: val_year=2014, AUC=0.7652
Fold3: val_year=2015, AUC=0.7513
Fold4: val_year=2016, AUC=0.7633

CV AUC (fold mean): 0.7600 ± 0.0053


In [16]:
# 提出用: 全 train で 1 本学習して test を予測（ベースラインと同じく「全データで学習→予測」）
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv (rows: {len(submission):,}) [全trainで学習した1本のモデルで予測]")

# 特徴量重要度（model_full の gain）
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

Saved submission.csv (rows: 40,716) [全trainで学習した1本のモデルで予測]


,feature,importance
7,directors,727
8,authors,601
0,rotten_tomatoes_link,488
4,movie_title,411
1,critic_name,211
3,publisher_name,155
9,actors,93
11,production_company,83
27,critic_name_te_ts,54
5,movie_info,41



重要度 Top5: ['directors', 'authors', 'rotten_tomatoes_link', 'movie_title', 'critic_name']
